# LeetCode 1238 — Circular Permutation in Binary Representation bằng Python

Link đề: https://leetcode.com/problems/circular-permutation-in-binary-representation/ · Độ khó: **Medium** · Chủ đề: Math, Backtracking, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def circularPermutation(self, n: int, start: int) -> List[int]:
        ...
```

Ghi chú: bài có **nhiều đáp án đúng** (đề nói "return any permutation"), nên khi kiểm thử ta dùng một **hàm kiểm tra hợp lệ** thay vì so khớp với một danh sách cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Đọc/viết số ở dạng nhị phân trong Python (`bin`, `format`, `^`, `>>`, `bit_count`) và hiểu "hai số khác nhau đúng một bit" nghĩa là `a ^ b` là một **lũy thừa của 2**.
2. Nhận ra bài toán là tìm **chu trình Hamilton** (đi qua mỗi đỉnh đúng một lần rồi quay về) trên **khối lập phương n chiều** (hypercube), và giải bằng **backtracking** cho n nhỏ.
3. Biết **mã Gray** (Gray code) `g(i) = i ^ (i >> 1)` và cách dựng nó bằng phép **phản chiếu** (reflect).
4. Hiểu mẹo then chốt: **XOR mọi phần tử với `start`** giữ nguyên tính chất "khác một bit" → đáp án `start ^ i ^ (i >> 1)`.
5. Viết hàm kiểm tra tính hợp lệ của một hoán vị vòng để test lời giải có nhiều đáp án.

## 2. Đề bài & ví dụ

Cho hai số nguyên `n` và `start`. Trả về **một hoán vị bất kỳ** `p` của `(0, 1, 2, …, 2ⁿ − 1)` sao cho:

- `p[0] = start`;
- `p[i]` và `p[i+1]` khác nhau **đúng một bit** trong biểu diễn nhị phân;
- `p[0]` và `p[2ⁿ − 1]` cũng khác nhau đúng một bit (tức là **vòng tròn** khép kín).

**Ví dụ 1:** `n = 2, start = 3` → `[3,2,0,1]` (nhị phân `11, 10, 00, 01`). Một đáp án hợp lệ khác: `[3,1,0,2]`.

**Ví dụ 2:** `n = 3, start = 2` → `[2,6,7,5,4,0,1,3]` (nhị phân `010,110,111,101,100,000,001,011`).

**Ràng buộc:** `1 <= n <= 16`, `0 <= start < 2ⁿ`.

In [1]:
import itertools
import random
import time
from typing import List

examples = [(2, 3, [3, 2, 0, 1]), (3, 2, [2, 6, 7, 5, 4, 0, 1, 3])]
for n, start, sample in examples:
    print(n, start, sample, [format(v, f"0{n}b") for v in sample])

2 3 [3, 2, 0, 1] ['11', '10', '00', '01']
3 2 [2, 6, 7, 5, 4, 0, 1, 3] ['010', '110', '111', '101', '100', '000', '001', '011']


## 3. Kiến thức nền

### 3.1 Biểu diễn nhị phân trong Python

- `bin(5)` → `'0b101'`; `format(5, '04b')` → `'0101'` (đệm 0 cho đủ 4 chữ số).
- `1 << k` = 2ᵏ, là số chỉ có bit thứ `k` bật.
- `x >> 1` dịch phải 1 bit (chia nguyên cho 2).

In [2]:
for x in [0, 1, 2, 5, 7]:
    print(x, bin(x), format(x, "04b"), "x>>1 =", x >> 1)
print([1 << k for k in range(5)])

0 0b0 0000 x>>1 = 0
1 0b1 0001 x>>1 = 0
2 0b10 0010 x>>1 = 1
5 0b101 0101 x>>1 = 2
7 0b111 0111 x>>1 = 3
[1, 2, 4, 8, 16]


### 3.2 Phép XOR `^` và "khác đúng một bit"

`a ^ b` bật đúng những bit mà `a` và `b` khác nhau. Vậy `a`, `b` khác nhau đúng một bit ⇔ `a ^ b` có đúng một bit 1 ⇔ `a ^ b` là lũy thừa của 2.

Kiểm tra "là lũy thừa của 2" cho `d > 0`: `d & (d - 1) == 0` (bỏ bit 1 thấp nhất thì còn 0). Hoặc `d.bit_count() == 1` (Python ≥ 3.10), hoặc `bin(d).count('1') == 1`.

In [3]:
def one_bit_diff(a, b):
    d = a ^ b
    return d != 0 and d & (d - 1) == 0


for a, b in [(3, 2), (2, 0), (0, 1), (3, 0), (5, 4), (6, 5)]:
    print(f"{a}={a:03b}, {b}={b:03b}, a^b={a ^ b:03b}, khác 1 bit? {one_bit_diff(a, b)}")

3=011, 2=010, a^b=001, khác 1 bit? True
2=010, 0=000, a^b=010, khác 1 bit? True
0=000, 1=001, a^b=001, khác 1 bit? True
3=011, 0=000, a^b=011, khác 1 bit? False
5=101, 4=100, a^b=001, khác 1 bit? True
6=110, 5=101, a^b=011, khác 1 bit? False


### 3.3 Tính chất đại số của XOR

- `a ^ a = 0`, `a ^ 0 = a`, giao hoán và kết hợp.
- **Quan trọng:** `(a ^ s) ^ (b ^ s) = a ^ b`. Nghĩa là XOR cùng một số `s` vào cả hai **không làm đổi** chỗ khác nhau giữa chúng.
- Ánh xạ `x ↦ x ^ s` là một **song ánh** (bijection) trên `[0, 2ⁿ)`: nó tự là nghịch đảo của chính nó.

In [4]:
s = 0b101
for a, b in [(3, 2), (6, 7), (0, 4)]:
    print((a ^ s) ^ (b ^ s) == a ^ b, end=" ")
print()
n = 3
print(sorted(x ^ s for x in range(1 << n)) == list(range(1 << n)))

True True True 
True


### 3.4 Hàm kiểm tra hợp lệ

Vì có nhiều đáp án, ta viết `is_valid(p, n, start)` kiểm tra đủ bốn điều: độ dài `2ⁿ`, là hoán vị của `0..2ⁿ−1`, `p[0] == start`, mọi cặp kề (kể cả cặp cuối–đầu) khác đúng một bit.

In [5]:
def is_valid(p, n, start):
    N = 1 << n
    if len(p) != N or p[0] != start or sorted(p) != list(range(N)):
        return False
    return all(one_bit_diff(p[i], p[(i + 1) % N]) for i in range(N))


print(is_valid([3, 2, 0, 1], 2, 3), is_valid([3, 1, 0, 2], 2, 3), is_valid([3, 0, 1, 2], 2, 3))
print(is_valid([2, 6, 7, 5, 4, 0, 1, 3], 3, 2))

True True False
True


## 4. Ý tưởng nền tảng / trực giác

Hình dung mỗi số `0..2ⁿ−1` là một **đỉnh**, nối cạnh giữa hai số khác đúng một bit. Với `n = 2` ta được một hình vuông; `n = 3` là hình lập phương:

```
n = 2:   00 — 01        n = 3: mỗi đỉnh có 3 hàng xóm (lật 1 trong 3 bit)
         |     |
         10 — 11
```

Đề yêu cầu đi vòng qua **mọi đỉnh đúng một lần** rồi quay về điểm xuất phát — chu trình Hamilton. Với `n = 2, start = 3 (11)`: đi `11 → 10 → 00 → 01 → (11)` là chạy quanh hình vuông.

Tính tay danh sách hàng xóm của `3` khi `n = 3`: lật bit 0 → `2`, bit 1 → `1`, bit 2 → `7`.

In [6]:
def neighbors(x, n):
    return [x ^ (1 << k) for k in range(n)]


print("n=2:", {x: neighbors(x, 2) for x in range(4)})
print("hàng xóm của 3 khi n=3:", neighbors(3, 3))

n=2: {0: [1, 2], 1: [0, 3], 2: [3, 0], 3: [2, 1]}
hàng xóm của 3 khi n=3: [2, 1, 7]


## 5. Cách 1 — Brute Force / baseline

Thử **mọi hoán vị** bắt đầu bằng `start` của các số còn lại, trả về hoán vị đầu tiên hợp lệ. Có `(2ⁿ − 1)!` hoán vị — chỉ khả thi với `n ≤ 3` (7! = 5040).

In [7]:
class SolutionBrute:
    def circularPermutation(self, n, start):
        rest = [x for x in range(1 << n) if x != start]
        for perm in itertools.permutations(rest):
            p = [start, *perm]
            if is_valid(p, n, start):
                return p
        return []


for n, start, _ in examples:
    p = SolutionBrute().circularPermutation(n, start)
    print(n, start, p, is_valid(p, n, start))

2 3 [3, 1, 0, 2] True
3 2 [2, 0, 1, 3, 7, 5, 4, 6] True


**Độ phức tạp:** `O((2ⁿ)! · 2ⁿ)` — với `n = 4` đã là `15! ≈ 1.3·10¹²`. **Nút thắt:** sinh cả những hoán vị sai ngay từ bước thứ hai (hai số kề không khác một bit). Cần **cắt nhánh**: chỉ đi tới hàng xóm hợp lệ.

## 6. Bước đột phá

**Insight 1 — Mã Gray:** dãy `g(i) = i ^ (i >> 1)` với `i = 0..2ⁿ−1` là **mã Gray phản chiếu** (reflected binary Gray code): hai phần tử kề nhau (kể cả `g(2ⁿ−1)` và `g(0) = 0`) khác đúng một bit. Đây là một chu trình Hamilton xuất phát từ `0`.

**Insight 2 — Dời gốc bằng XOR:** nếu `q` là một chu trình hợp lệ bắt đầu từ `0`, thì `p[i] = q[i] ^ start`:

- `p[0] = 0 ^ start = start` ✓
- `p[i] ^ p[i+1] = q[i] ^ q[i+1]` (XOR cùng `start` triệt tiêu) → vẫn khác một bit ✓
- `x ↦ x ^ start` là song ánh → `p` vẫn là hoán vị ✓

**Quy tắc:** "Nếu cần chu trình Gray bắt đầu ở `start` thì lấy Gray bắt đầu ở 0 rồi XOR với `start`."

→ Đáp án một dòng: `[start ^ i ^ (i >> 1) for i in range(1 << n)]`.

In [8]:
n = 3
g = [i ^ (i >> 1) for i in range(1 << n)]
print("Gray:", g, [format(v, "03b") for v in g])
print("Gray hợp lệ từ 0:", is_valid(g, n, 0))
p = [x ^ 2 for x in g]
print("XOR 2:", p, is_valid(p, n, 2))

Gray: [0, 1, 3, 2, 6, 7, 5, 4] ['000', '001', '011', '010', '110', '111', '101', '100']
Gray hợp lệ từ 0: True
XOR 2: [2, 3, 1, 0, 4, 5, 7, 6] True


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Backtracking (DFS + cắt nhánh)

Từ đỉnh hiện tại, thử lần lượt các hàng xóm (lật 1 bit) chưa thăm; nếu đi hết `2ⁿ` đỉnh và đỉnh cuối kề `start` thì thành công, ngược lại quay lui. Trên hypercube, thử lật bit **thấp nhất trước** thì thực tế không bao giờ phải quay lui (nó tái tạo đúng mã Gray), nên chạy nhanh. Dùng stack tường minh để tránh đệ quy sâu `2¹⁶`.

In [9]:
class SolutionBacktrack:
    def circularPermutation(self, n, start):
        N = 1 << n
        path = [start]
        used = [False] * N
        used[start] = True
        it = [0]                      # it[d] = bit tiếp theo sẽ thử ở độ sâu d
        while path:
            if len(path) == N:
                if one_bit_diff(path[-1], start):
                    return path
                # không khép vòng -> quay lui
                used[path.pop()] = False
                it.pop()
                continue
            cur = path[-1]
            moved = False
            while it[-1] < n:
                k = it[-1]
                it[-1] += 1
                nxt = cur ^ (1 << k)
                if not used[nxt]:
                    used[nxt] = True
                    path.append(nxt)
                    it.append(0)
                    moved = True
                    break
            if not moved:             # hết lựa chọn -> quay lui
                used[path.pop()] = False
                it.pop()
        return []


for n, start, _ in examples:
    p = SolutionBacktrack().circularPermutation(n, start)
    print(n, start, p, is_valid(p, n, start))

2 3 [3, 2, 0, 1] True
3 2 [2, 3, 1, 0, 4, 5, 7, 6] True


**Độ phức tạp:** trường hợp xấu của backtracking tổng quát là hàm mũ, nhưng với thứ tự "bit thấp trước" trên hypercube, mỗi đỉnh được thêm đúng một lần và thử tối đa `n` bit → `O(n · 2ⁿ)`. Bộ nhớ `O(2ⁿ)`. Ta sẽ kiểm chứng bằng test tới `n = 16`.

### 7.2 Cách 3 — Mã Gray rồi **xoay vòng** tới `start`

Sinh Gray `g` từ 0, tìm vị trí `k` có `g[k] == start`, rồi xoay: `g[k:] + g[:k]`. Vì `g` là một **vòng** khép kín, xoay vẫn giữ mọi cặp kề.

In [10]:
class SolutionRotate:
    def circularPermutation(self, n, start):
        g = [i ^ (i >> 1) for i in range(1 << n)]
        k = g.index(start)
        return g[k:] + g[:k]


for n, start, _ in examples:
    p = SolutionRotate().circularPermutation(n, start)
    print(n, start, p, is_valid(p, n, start))

2 3 [3, 2, 0, 1] True
3 2 [2, 6, 7, 5, 4, 0, 1, 3] True


**Độ phức tạp:** `O(2ⁿ)` thời gian và bộ nhớ. Đáp án **khác** Cách 4 nhưng đều hợp lệ.

### 7.3 Cách 4 — XOR với `start` (một dòng)

In [11]:
class SolutionXor:
    def circularPermutation(self, n, start):
        return [start ^ i ^ (i >> 1) for i in range(1 << n)]


for n, start, sample in examples:
    p = SolutionXor().circularPermutation(n, start)
    print(n, start, p, is_valid(p, n, start), "| trùng ví dụ đề?", p == sample)

2 3 [3, 2, 0, 1] True | trùng ví dụ đề? True
3 2 [2, 3, 1, 0, 4, 5, 7, 6] True | trùng ví dụ đề? False


Thú vị: với ví dụ 1, kết quả `[3,2,0,1]` trùng đúng ví dụ của đề — có lẽ đề được sinh bằng chính công thức này.

**Độ phức tạp:** `O(2ⁿ)` thời gian, `O(1)` bộ nhớ phụ ngoài kết quả.

### 7.4 Cách 5 — Dựng Gray bằng phản chiếu (reflect), rồi XOR

Mã Gray `n` bit dựng từ `n − 1` bit: lấy dãy cũ `G`, nối với **bản đảo ngược** của `G` mà mỗi phần tử bật thêm bit cao `1 << (n−1)`. Chỗ nối và chỗ khép vòng đều chỉ khác bit cao.

```
G1 = [0, 1]
G2 = [00, 01] + [11, 10] = [0, 1, 3, 2]
G3 = [0, 1, 3, 2] + [6, 7, 5, 4]
```

In [12]:
class SolutionReflect:
    def circularPermutation(self, n, start):
        g = [0]
        for k in range(n):
            high = 1 << k
            g += [x | high for x in reversed(g)]
        return [x ^ start for x in g]


for n, start, _ in examples:
    p = SolutionReflect().circularPermutation(n, start)
    print(n, start, p, is_valid(p, n, start))
print("reflect == công thức i^(i>>1):",
      SolutionReflect().circularPermutation(5, 0) == [i ^ (i >> 1) for i in range(32)])

2 3 [3, 2, 0, 1] True
3 2 [2, 3, 1, 0, 4, 5, 7, 6] True
reflect == công thức i^(i>>1): True


**Độ phức tạp:** tổng độ dài các lần nối `1 + 2 + … + 2ⁿ⁻¹ < 2ⁿ` → `O(2ⁿ)`.

## 8. TRACE từng bước

Trace Cách 4 với `n = 3, start = 2 (010)`:

| i | i (bin) | i>>1 | g = i^(i>>1) | p = g^start | p (bin) | bit đổi so với trước |
|---|---|---|---|---|---|---|
| 0 | 000 | 000 | 000 | 2 | 010 | — |
| 1 | 001 | 000 | 001 | 3 | 011 | bit 0 |
| 2 | 010 | 001 | 011 | 1 | 001 | bit 1 |
| 3 | 011 | 001 | 010 | 0 | 000 | bit 0 |
| 4 | 100 | 010 | 110 | 4 | 100 | bit 2 |
| 5 | 101 | 010 | 111 | 5 | 101 | bit 0 |
| 6 | 110 | 011 | 101 | 7 | 111 | bit 1 |
| 7 | 111 | 011 | 100 | 6 | 110 | bit 0 |

Cặp khép vòng: `6 (110)` và `2 (010)` khác bit 2. Nhận xét: bit đổi ở bước `i` chính là **vị trí bit 1 thấp nhất của `i`** (dãy 0,1,0,2,0,1,0 — "dãy thước kẻ").

In [13]:
def trace_xor(n, start):
    N = 1 << n
    p = []
    for i in range(N):
        g = i ^ (i >> 1)
        v = g ^ start
        if p:
            d = p[-1] ^ v
            changed = f"bit {d.bit_length() - 1}"
        else:
            changed = "—"
        p.append(v)
        print(f"i={i:{n}b}  i>>1={i >> 1:0{n}b}  g={g:0{n}b}  p={v:2} ({v:0{n}b})  đổi: {changed}")
    d = p[-1] ^ p[0]
    print(f"khép vòng: {p[-1]} -> {p[0]} đổi bit {d.bit_length() - 1}")
    return p


trace_xor(3, 2)

i=  0  i>>1=000  g=000  p= 2 (010)  đổi: —
i=  1  i>>1=000  g=001  p= 3 (011)  đổi: bit 0
i= 10  i>>1=001  g=011  p= 1 (001)  đổi: bit 1
i= 11  i>>1=001  g=010  p= 0 (000)  đổi: bit 0
i=100  i>>1=010  g=110  p= 4 (100)  đổi: bit 2
i=101  i>>1=010  g=111  p= 5 (101)  đổi: bit 0
i=110  i>>1=011  g=101  p= 7 (111)  đổi: bit 1
i=111  i>>1=011  g=100  p= 6 (110)  đổi: bit 0
khép vòng: 6 -> 2 đổi bit 2


[2, 3, 1, 0, 4, 5, 7, 6]

Trace thêm Cách 2 (backtracking) với `n = 2, start = 3` để thấy nó chọn bit thấp trước:

In [14]:
def trace_backtrack(n, start):
    N = 1 << n
    path, used = [start], {start}
    while len(path) < N:
        cur = path[-1]
        for k in range(n):
            nxt = cur ^ (1 << k)
            if nxt not in used:
                print(f"tại {cur:0{n}b}: lật bit {k} -> {nxt:0{n}b}")
                used.add(nxt)
                path.append(nxt)
                break
        else:
            print("kẹt (cần quay lui)")
            break
    print("path:", path, "| hợp lệ:", is_valid(path, n, start))


trace_backtrack(2, 3)

tại 11: lật bit 0 -> 10
tại 10: lật bit 1 -> 00
tại 00: lật bit 0 -> 01
path: [3, 2, 0, 1] | hợp lệ: True


## 9. Code nộp LeetCode

In [15]:
from typing import List


class Solution:
    def circularPermutation(self, n: int, start: int) -> List[int]:
        return [start ^ i ^ (i >> 1) for i in range(1 << n)]


for n, start, _ in examples:
    assert is_valid(Solution().circularPermutation(n, start), n, start)
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao đúng

**Bổ đề (Gray):** với `g(i) = i ^ (i >> 1)`, ta có `g(i) ^ g(i+1)` là lũy thừa của 2.

Chứng minh: `g(i) ^ g(i+1) = (i ^ (i+1)) ^ ((i ^ (i+1)) >> 1)` (vì `>>` phân phối qua `^`). Khi cộng 1, `i ^ (i+1)` có dạng `0…01…1` (gồm `t + 1` bit 1 ở cuối, với `t` = số bit 1 liên tiếp ở cuối `i`). Gọi `m = 2^(t+1) − 1`, khi đó `m ^ (m >> 1) = 2^t` — đúng một bit. ∎

**Khép vòng:** `g(0) = 0` và `g(2ⁿ − 1) = (2ⁿ − 1) ^ (2ⁿ⁻¹ − 1) = 2ⁿ⁻¹` — khác đúng bit cao nhất.

**Hoán vị:** `g` là song ánh trên `[0, 2ⁿ)` vì có nghịch đảo (`i = g ^ (g >> 1) ^ (g >> 2) ^ …`); kết hợp với song ánh `x ↦ x ^ start`.

**Bắt đầu đúng:** `p[0] = start ^ 0 = start`.

Ghép lại với Insight 2 (mục 6): `p` thoả mãn cả ba điều kiện. Ta còn kiểm chứng bằng máy cho mọi `n ≤ 16` và nhiều `start` ở mục 12.

In [16]:
# Kiểm chứng bổ đề và nghịch đảo của Gray với n = 10
n = 10
g = [i ^ (i >> 1) for i in range(1 << n)]
assert all(one_bit_diff(g[i], g[i + 1]) for i in range((1 << n) - 1))
assert g[-1] == 1 << (n - 1)


def gray_inverse(v):
    i = 0
    while v:
        i ^= v
        v >>= 1
    return i


assert all(gray_inverse(g[i]) == i for i in range(1 << n))
print("bổ đề Gray + nghịch đảo OK với n = 10")

bổ đề Gray + nghịch đảo OK với n = 10


## 11. Time / Space Complexity

Đặt `N = 2ⁿ` (độ dài đáp án, tối đa `65536`).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute hoán vị | `O(N! · N)` | `O(N)` |
| 2. Backtracking (bit thấp trước) | `O(n · N)` thực tế | `O(N)` |
| 3. Gray + xoay | `O(N)` | `O(N)` (bản sao khi xoay) |
| 4. XOR công thức | `O(N)` | `O(1)` |
| 5. Phản chiếu + XOR | `O(N)` | `O(N)` |

Đáp án luôn cần `O(N)` bộ nhớ, nên mọi cách 2–5 đều tối ưu về bậc.

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [17]:
ALL = [SolutionBacktrack, SolutionRotate, SolutionXor, SolutionReflect, Solution]

for n, start, _ in examples:
    for cls in ALL + [SolutionBrute]:
        assert is_valid(cls().circularPermutation(n, start), n, start), cls.__name__

# n = 1: chỉ có [start, 1 - start]
for start in (0, 1):
    for cls in ALL:
        assert cls().circularPermutation(1, start) == [start, 1 - start]

# start = 0 và start = 2^n - 1
for n in range(1, 8):
    for start in (0, (1 << n) - 1):
        for cls in ALL:
            assert is_valid(cls().circularPermutation(n, start), n, start)
print("edge cases OK")

edge cases OK


### 12.2 Phản ví dụ cho hàm kiểm tra

Hàm `is_valid` phải bắt được: sai `start`, thiếu phần tử, cặp kề khác 2 bit, không khép vòng.

In [18]:
assert not is_valid([0, 1, 3, 2], 2, 3)            # sai start
assert not is_valid([3, 2, 0], 2, 3)               # thiếu phần tử
assert not is_valid([3, 0, 1, 2], 2, 3)            # 3 -> 0 khác 2 bit
assert not is_valid([0, 1, 3, 7, 6, 4, 5, 2], 3, 0)  # 5 -> 2 khác 3 bit, 2 -> 0 khép được
assert not is_valid([1, 3, 2, 6, 7, 5, 4, 0, 0], 3, 1)
print("is_valid bắt đúng các phản ví dụ")

is_valid bắt đúng các phản ví dụ


### 12.3 Ngẫu nhiên có seed + đối chiếu với brute force (n ≤ 3)

Brute force chỉ chạy được với `n ≤ 3`; với các `n` lớn hơn ta dựa vào `is_valid` (chính là "định nghĩa" của đáp án).

In [19]:
for n in (1, 2, 3):
    for start in range(1 << n):
        assert SolutionBrute().circularPermutation(n, start)   # brute luôn tìm thấy
        for cls in ALL:
            assert is_valid(cls().circularPermutation(n, start), n, start)

rng = random.Random(1238)
for _ in range(300):
    n = rng.randint(1, 10)
    start = rng.randrange(1 << n)
    for cls in ALL:
        assert is_valid(cls().circularPermutation(n, start), n, start)
print("300 test ngẫu nhiên OK")

300 test ngẫu nhiên OK


### 12.4 Test giới hạn `n = 16`

In [20]:
for cls in ALL:
    t = time.perf_counter()
    p = cls().circularPermutation(16, 12345)
    dt = time.perf_counter() - t
    assert is_valid(p, 16, 12345)
    print(f"{cls.__name__:18} n=16: {len(p)} phần tử, {dt * 1000:.1f} ms")

SolutionBacktrack  n=16: 65536 phần tử, 19.1 ms


SolutionRotate     n=16: 65536 phần tử, 4.1 ms


SolutionXor        n=16: 65536 phần tử, 4.5 ms


SolutionReflect    n=16: 65536 phần tử, 3.9 ms


Solution           n=16: 65536 phần tử, 4.4 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ trả về mã Gray từ 0, quên `start`

In [21]:
def wrong_forget_start(n, start):
    return [i ^ (i >> 1) for i in range(1 << n)]


p = wrong_forget_start(2, 3)
print("sai :", p, is_valid(p, 2, 3))
p = Solution().circularPermutation(2, 3)
print("đúng:", p, is_valid(p, 2, 3))

sai : [0, 1, 3, 2] False
đúng: [3, 2, 0, 1] True


**Vì sao sai:** `p[0]` phải là `start`. **Sửa:** XOR từng phần tử với `start` (hoặc xoay vòng tới `start`).

### Lỗi 2 — "Cộng" `start` thay vì XOR (dịch vòng theo giá trị)

Ý tưởng sai: `(g + start) % 2ⁿ` — tưởng là "dời gốc".

In [22]:
def wrong_add(n, start):
    N = 1 << n
    return [((i ^ (i >> 1)) + start) % N for i in range(N)]


p = wrong_add(2, 3)
print("sai :", p, [format(v, "02b") for v in p], is_valid(p, 2, 3))

sai : [3, 0, 2, 1] ['11', '00', '10', '01'] False


**Vì sao sai:** cộng làm thay đổi **nhiều bit** do có nhớ (carry): `0 + 3 = 3 (11)` rồi `1 + 3 = 4 % 4 = 0 (00)` khác 2 bit. Chỉ XOR mới bảo toàn hiệu `a ^ b`. **Sửa:** dùng `^`.

### Lỗi 3 — Dãy "khác một bit" nhưng không khép vòng

Ví dụ dùng **mã nhị phân thường** sắp sẵn rồi đảo một vài chỗ, hoặc sinh Gray rồi lại sắp xếp.

In [23]:
def wrong_sorted(n, start):
    g = sorted(start ^ i ^ (i >> 1) for i in range(1 << n))
    k = g.index(start)
    return g[k:] + g[:k]


p = wrong_sorted(3, 2)
print("sai :", p, is_valid(p, 3, 2))
bad = [(p[i], p[(i + 1) % 8]) for i in range(8) if not one_bit_diff(p[i], p[(i + 1) % 8])]
print("các cặp hỏng:", bad)

sai : [2, 3, 4, 5, 6, 7, 0, 1] False
các cặp hỏng: [(3, 4), (5, 6), (7, 0), (1, 2)]


**Vì sao sai:** sắp xếp phá cấu trúc Gray: `3 (011) → 4 (100)` khác 3 bit. **Sửa:** giữ nguyên thứ tự Gray, không sort.

### Lỗi 4 — Backtracking đệ quy với `n = 16` → tràn stack

In [24]:
import sys


def backtrack_recursive(n, start):
    N = 1 << n
    path, used = [start], {start}

    def dfs():
        if len(path) == N:
            return one_bit_diff(path[-1], start)
        for k in range(n):
            nxt = path[-1] ^ (1 << k)
            if nxt not in used:
                used.add(nxt)
                path.append(nxt)
                if dfs():
                    return True
                used.discard(path.pop())
        return False

    return path if dfs() else []


try:
    backtrack_recursive(12, 0)          # cần độ sâu 4096 > giới hạn mặc định
    print("chạy được")
except RecursionError:
    print("RecursionError với n = 12, giới hạn =", sys.getrecursionlimit())
print("n = 8 vẫn ổn:", is_valid(backtrack_recursive(8, 5), 8, 5))

RecursionError với n = 12, giới hạn = 1000
n = 8 vẫn ổn: True


**Vì sao sai:** độ sâu đệ quy bằng `2ⁿ` (tới 65536) vượt giới hạn đệ quy của Python (~1000). **Sửa:** dùng stack tường minh (Cách 2) hoặc — tốt nhất — công thức Gray `O(N)`.

## 14. Mẹo & nhận dạng pattern

- Thấy "các số `0..2ⁿ−1`, hai số kề khác đúng **một bit**" → nghĩ ngay **mã Gray** `i ^ (i >> 1)`.
- Cần bắt đầu từ một giá trị tuỳ ý → **XOR với `start`** (dời gốc trên hypercube) hoặc xoay vòng.
- "Khác đúng một bit" ⇔ `d = a ^ b` khác 0 và `d & (d - 1) == 0`.
- Bài "nhiều đáp án đúng" → viết **validator** trước, test bằng validator.
- Mã Gray dựng bằng **phản chiếu**: `G(n) = G(n−1) + reverse(G(n−1)) | (1 << (n−1))`.
- Bài liên quan: **89 Gray Code** (đúng bài này với `start = 0`), **753 Cracking the Safe** (chu trình de Bruijn — họ hàng của Gray).

## 15. Biến thể

1. **LeetCode 89 — Gray Code:** `start = 0` → `[i ^ (i >> 1) for i in range(1 << n)]`.
2. **Tìm vị trí của `x` trong dãy Gray** → nghịch đảo Gray (`gray_inverse` ở mục 10).
3. **Không cần khép vòng** (chỉ đường đi Hamilton): mọi lời giải trên vẫn dùng được.
4. **Đếm số bit đổi của từng vị trí** (bit 0 đổi `2ⁿ⁻¹` lần...) → dãy thước kẻ.
5. **Gray code k-ary** (mỗi chữ số từ `0..k-1`, kề nhau khác một chữ số đúng 1 đơn vị) → phản chiếu tổng quát.

Demo biến thể 2: tìm chỉ số của `x` trong đáp án của bài (đáp án Cách 4) mà không dựng cả dãy.

In [25]:
def index_in_answer(n, start, x):
    # p[i] = start ^ g(i) = x  <=>  g(i) = x ^ start  <=>  i = gray_inverse(x ^ start)
    return gray_inverse(x ^ start)


n, start = 5, 13
p = Solution().circularPermutation(n, start)
assert all(p[index_in_answer(n, start, x)] == x for x in range(1 << n))
print("index_in_answer OK; ví dụ x = 0 ở vị trí", index_in_answer(n, start, 0))

index_in_answer OK; ví dụ x = 0 ở vị trí 9


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute hoán vị | Thử mọi hoán vị | `O(N! · N)` | `O(N)` | Chỉ để kiểm thử `n ≤ 3` |
| 2. Backtracking | DFS lật bit, quay lui | `O(n · N)` thực tế | `O(N)` | Khi chưa biết mã Gray; bài Hamilton tổng quát |
| 3. Gray + xoay | Sinh Gray, xoay tới `start` | `O(N)` | `O(N)` | Dễ nghĩ ra nếu đã biết Gray |
| 4. XOR công thức | `start ^ i ^ (i >> 1)` | `O(N)` | `O(1)` phụ | Lời giải nộp — ngắn và nhanh nhất |
| 5. Phản chiếu + XOR | Dựng Gray đệ quy rồi XOR | `O(N)` | `O(N)` | Khi muốn hiểu vì sao Gray đúng |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `Solution().circularPermutation(2, 0)` trả về gì?

**Bài 2 (tính tay):** tính `g(i) = i ^ (i >> 1)` cho `i = 12` (nhị phân `1100`).

**Bài 3 (điền TODO):** viết `gray_code(n)` trả về mã Gray `n` bit bắt đầu từ 0 bằng phương pháp **phản chiếu** (không dùng công thức `i ^ (i >> 1)`).

**Bài 4 (tự viết lại):** viết `circular_by_rotation(n, start)` bằng cách xoay vòng, **không** dùng `list.index` (gợi ý: dùng `gray_inverse`).

**Bài 5 (mini challenge):** viết `count_valid_rotations(p)`: cho một hoán vị vòng hợp lệ `p` (bắt đầu tại `p[0]`), đếm xem có bao nhiêu `start` khác nhau mà phép xoay `p` thành bắt đầu tại `start` vẫn hợp lệ. (Đáp án phải là `len(p)` — hãy viết hàm kiểm chứng điều đó.)

In [26]:
def gray_code(n):
    # TODO: phản chiếu: g = [0]; mỗi bước g += [x | (1 << k) for x in reversed(g)]
    raise NotImplementedError


def circular_by_rotation(n, start):
    # TODO: k = gray_inverse(start); trả về Gray xoay bắt đầu tại k
    raise NotImplementedError


def count_valid_rotations(p):
    # TODO
    raise NotImplementedError


# Test gợi ý:
# assert gray_code(3) == [0, 1, 3, 2, 6, 7, 5, 4]
# assert is_valid(circular_by_rotation(4, 9), 4, 9)
# assert count_valid_rotations(Solution().circularPermutation(3, 2)) == 8

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: `start = 0` nên đáp án chính là mã Gray.
- Mức 2: Gray 2 bit: `00, 01, 11, 10`.
- Mức 3: `12 = 1100`, `12 >> 1 = 0110`, XOR → `1010 = 10`.

**Bài 3–4:**
- Mức 1: xem Cách 5 (mục 7.4) và hàm `gray_inverse` (mục 10).
- Mức 2: vị trí của `start` trong Gray là `gray_inverse(start)`.
- Mức 3: `[g((k + i) % N) for i in range(N)]` với `g(i) = i ^ (i >> 1)`.

**Bài 5:**
- Mức 1: xoay một vòng không làm đổi tập các cặp kề.
- Mức 2: thử mọi `k`, dựng `p[k:] + p[:k]`, kiểm tra `is_valid`.
- Mức 3: đếm số `k` hợp lệ.

## 19. Đáp án tham khảo

**Bài 1:** `[0, 1, 3, 2]`. **Bài 2:** `10`.

In [27]:
assert Solution().circularPermutation(2, 0) == [0, 1, 3, 2]
assert 12 ^ (12 >> 1) == 10
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [28]:
def gray_code(n):
    g = [0]
    for k in range(n):
        g += [x | (1 << k) for x in reversed(g)]
    return g


def circular_by_rotation(n, start):
    N = 1 << n
    k = gray_inverse(start)
    return [((k + i) % N) ^ (((k + i) % N) >> 1) for i in range(N)]


def count_valid_rotations(p):
    N = len(p)
    n = N.bit_length() - 1
    return sum(is_valid(p[k:] + p[:k], n, p[k]) for k in range(N))


assert gray_code(3) == [0, 1, 3, 2, 6, 7, 5, 4]
assert all(gray_code(n) == [i ^ (i >> 1) for i in range(1 << n)] for n in range(0, 11))
rng = random.Random(42)
for _ in range(100):
    n = rng.randint(1, 9)
    start = rng.randrange(1 << n)
    assert is_valid(circular_by_rotation(n, start), n, start)
assert count_valid_rotations(Solution().circularPermutation(3, 2)) == 8
assert count_valid_rotations(Solution().circularPermutation(5, 7)) == 32
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
class Solution:
    def circularPermutation(self, n: int, start: int) -> List[int]:
        return [start ^ i ^ (i >> 1) for i in range(1 << n)]
```

Ghi nhớ: **Gray(i) = i ^ (i >> 1)**; **dời gốc = XOR start**; **khác 1 bit ⇔ d & (d−1) == 0, d ≠ 0**.

**Checklist trước khi submit:**

- [ ] Dùng `1 << n` (= `2ⁿ`), không nhầm `2 ^ n` (trong Python `^` là XOR!).
- [ ] Phần tử đầu là `start`.
- [ ] Dời gốc bằng `^`, không bằng `+`.
- [ ] Không sort lại dãy Gray.
- [ ] Tránh đệ quy sâu `2ⁿ`; công thức `O(2ⁿ)` là đủ.
- [ ] Tự test bằng `is_valid` (nhiều đáp án đúng).